# The Chi-Squared Distribution

This notebook introduces the **chi-squared ($\chi^2$) distribution** through a communications example: the *power* of Gaussian noise. Squaring a standard normal gives $\chi^2$ with 1 degree of freedom; summing squares over $k$ transmissions gives $k$ degrees of freedom. It is based on the eleventh video of Lesson 2 (Probability Distributions).


## 1. Introduction

You transmit the bits $1\,0\,0\,1\,0$ through the air. The channel adds **noise** — from interfering devices (a Wi-Fi router), obstructions (walls, trees), weather (rain, humidity), electrical interference. What arrives is your message **plus noise** $Z$, random in nature.

A common assumption in communications: $Z$ is Gaussian with mean $\mu = 0$. A key quantity is the **noise power**, modeled by the square of the noise:

$$
W = Z^2
$$

Power relates to the variance (dispersion) of the noise and determines how hard it is to read the received signal. The question: **what is the distribution of $W$?**

After completing this notebook you will be able to:

- Derive the CDF of $W = Z^2$ from areas under the Gaussian.
- Explain why the $\chi^2_1$ PDF piles up near 0 (steep CDF = high density).
- Generalize to $k$ transmissions: $\chi^2_k = Z_1^2 + \dots + Z_k^2$, and describe how the shape changes with $k$.


## 2. Theory

### The CDF of $W = Z^2$, geometrically

Take $Z \sim N(0, 1)$ (standard normal, for simplicity). The event $W \leq w$ happens exactly when

$$
-\sqrt{w} \leq Z \leq \sqrt{w}
$$

— each value of $w$ is reached by **two** values of $Z$, namely $\pm\sqrt{w}$. So

$$
F_W(w) = P(W \leq w) = \text{area under the Gaussian between } -\sqrt{w} \text{ and } \sqrt{w}
$$

For small $w$ this area grows **fast** — the Gaussian concentrates its probability around 0. That steep start is the signature of the distribution.

### The PDF is the derivative of the CDF

Since the CDF is the accumulated area (the integral of the PDF), the PDF is its **derivative** — the slope at each point. The steep early CDF means a **large density near 0**, decaying as $w$ grows.

This distribution is called **chi-squared with 1 degree of freedom**, $\chi^2_1$.

### More transmissions, more degrees of freedom

Noise power accumulated over 2 transmissions: $Z_1^2 + Z_2^2 \sim \chi^2_2$. Over 5: $Z_1^2 + \cdots + Z_5^2 \sim \chi^2_5$. In general,

$$
Z_1^2 + Z_2^2 + \cdots + Z_k^2 \sim \chi^2_k
$$

for independent standard normals. As $k$ increases, the PDF **spreads out and becomes more symmetrical** — the peak moves away from 0.


## 3. Worked Example

For $\chi^2_1$, using the standard normal CDF $\Phi$:

$$
F_W(w) = \Phi(\sqrt{w}) - \Phi(-\sqrt{w}) = 2\Phi(\sqrt{w}) - 1
$$

- $F_W(1) = 2\Phi(1) - 1 \approx 0.683$ — the 1σ rule in disguise: noise power below 1 exactly means noise within $\pm 1$.
- $F_W(4) = 2\Phi(2) - 1 \approx 0.954$ — the 2σ rule: $W \leq 4 \iff |Z| \leq 2$.

Two-thirds of the probability sits below $w = 1$: small noise powers dominate.


## 4. Python Examples


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from math import erf, sqrt, gamma

rng = np.random.default_rng(seed=42)

def normal_pdf(x):
    return np.exp(-x**2 / 2) / np.sqrt(2 * np.pi)

@np.vectorize
def normal_cdf(x):
    return 0.5 * (1 + erf(x / sqrt(2)))

def chi2_cdf_1dof(w):
    return 2 * normal_cdf(np.sqrt(w)) - 1

def chi2_pdf(x, k):
    x = np.asarray(x, dtype=float)
    return np.where(x > 0,
                    x ** (k / 2 - 1) * np.exp(-x / 2) / (2 ** (k / 2) * gamma(k / 2)),
                    0.0)


### The CDF from Gaussian areas


In [ ]:
for w in [0.5, 1, 2, 4]:
    print(f"F_W({w}) = P(-√{w} <= Z <= √{w}) = {chi2_cdf_1dof(w):.4f}")


### Simulation: squaring noise


In [ ]:
z = rng.normal(0, 1, size=1_000_000)
w = z ** 2

print(f"Simulated P(W <= 1): {(w <= 1).mean():.4f}   theory: {chi2_cdf_1dof(1):.4f}")
print(f"Simulated P(W <= 4): {(w <= 4).mean():.4f}   theory: {chi2_cdf_1dof(4):.4f}")
print(f"Fraction of W below 0.1: {(w <= 0.1).mean():.3f} — density piles up near 0")


## 5. Visualizations

### The video's geometric construction

Left: the event $W \leq w$ is the Gaussian area between $-\sqrt{w}$ and $\sqrt{w}$. Right: collecting those areas for every $w$ draws the $\chi^2_1$ CDF — steep at first, then flattening.


In [ ]:
w0 = 1.0
xs = np.linspace(-4, 4, 500)
ws = np.linspace(0.001, 8, 400)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(xs, normal_pdf(xs), color="#4e79a7", linewidth=2)
mask = np.abs(xs) <= np.sqrt(w0)
axes[0].fill_between(xs[mask], normal_pdf(xs[mask]), color="#e15759", alpha=0.6)
for s in [-1, 1]:
    axes[0].axvline(s * np.sqrt(w0), color="gray", linestyle=":")
axes[0].set_title(f"P(W ≤ {w0:.0f}) = area between ±√{w0:.0f} = {chi2_cdf_1dof(w0):.3f}")
axes[0].set_xlabel("z (noise)")
axes[0].set_ylabel("Gaussian PDF")

axes[1].plot(ws, chi2_cdf_1dof(ws), color="#e15759", linewidth=2)
axes[1].plot(w0, chi2_cdf_1dof(w0), "ko")
axes[1].set_title("χ²₁ CDF: fast area gain near 0, then slower")
axes[1].set_xlabel("w (noise power)")
axes[1].set_ylabel("F(w)")

for ax in axes:
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### PDF = slope of the CDF

The numerical derivative of the CDF lands exactly on the $\chi^2_1$ density — huge near 0 (the steep part), then decaying.


In [ ]:
dw = ws[1] - ws[0]
slope = np.gradient(chi2_cdf_1dof(ws), dw)

plt.figure(figsize=(9, 5))
plt.hist(w[w <= 8], bins=120, density=True, color="#9ecae1",
         edgecolor="white", label="simulated W = Z²")
plt.plot(ws, slope, color="#e15759", linewidth=2.5, label="slope of the CDF")
plt.plot(ws, chi2_pdf(ws, 1), "k--", linewidth=1.5, label="χ²₁ PDF formula")
plt.ylim(0, 2)
plt.title("The PDF is the derivative of the CDF — density piles up near 0")
plt.xlabel("w (noise power)")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


### Degrees of freedom: power over k transmissions

Simulated sums of squares against the $\chi^2_k$ PDFs: as $k$ grows the distribution spreads and becomes more symmetrical.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
grid = np.linspace(0.01, 25, 500)

for ax, k in zip(axes, [1, 2, 5]):
    wk = (rng.normal(0, 1, size=(300_000, k)) ** 2).sum(axis=1)
    ax.hist(wk[wk <= 25], bins=100, density=True, color="#9ecae1",
            edgecolor="white")
    ax.plot(grid, chi2_pdf(grid, k), color="#e15759", linewidth=2)
    ax.set_title(f"χ² with k = {k} degrees of freedom")
    ax.set_xlabel("total noise power")
    ax.set_ylim(0, 1.2 if k == 1 else None)
    ax.grid(True, alpha=0.3)

axes[0].set_ylabel("Density")
plt.suptitle("Z₁² + ... + Z_k²: more transmissions → wider, more symmetric")
plt.tight_layout()
plt.show()


## 6. Exercises

### Basic

1. Which values can $W = Z^2$ take? Why is the $\chi^2$ PDF zero for negative arguments?
2. Express $P(W \leq 9)$ for $\chi^2_1$ in terms of the standard normal CDF, and evaluate it. Which σ-rule is this?

### Intermediate

3. For $\chi^2_1$, find the median noise power — the $w$ with $F_W(w) = 0.5$. (Hint: solve $2\Phi(\sqrt{w}) - 1 = 0.5$ numerically or by trial.)
4. If the noise is *not* standard — $Z \sim N(0, \sigma^2)$ — what simple transformation of $W = Z^2$ follows $\chi^2_1$?

### Advanced

5. Simulate $\chi^2_k$ for $k = 30$ and overlay a normal PDF with matching center ($k$) and spread ($\sqrt{2k}$). How good is the fit — and which theorem foreshadowed this?


## 7. Solutions


In [ ]:
# Exercise 1
print("1.  W = Z² >= 0 always — negative power is impossible, so the PDF is 0 there.")

# Exercise 2
print(f"2.  P(W<=9) = 2Φ(3) - 1 = {2*normal_cdf(3)-1:.4f} — the 3σ rule (99.7%).")

# Exercise 3: solve 2Φ(√w) - 1 = 0.5 numerically
grid = np.linspace(0.01, 3, 100_000)
median = grid[np.argmin(np.abs(chi2_cdf_1dof(grid) - 0.5))]
print(f"3.  median ≈ {median:.4f} (exact: (Φ⁻¹(0.75))² ≈ 0.4549) — "
      "well below the mean of 1")

# Exercise 4
print("4.  Z/σ is standard normal, so W/σ² = (Z/σ)² ~ χ²₁.")

# Exercise 5: chi-squared with k=30 vs a matched normal
k = 30
wk = (rng.normal(0, 1, size=(300_000, k)) ** 2).sum(axis=1)
grid = np.linspace(0, 70, 500)
normal_fit = np.exp(-(grid - k) ** 2 / (4 * k)) / np.sqrt(4 * np.pi * k)

plt.figure(figsize=(8, 4))
plt.hist(wk, bins=100, density=True, color="#9ecae1", edgecolor="white",
         label="simulated χ²₃₀")
plt.plot(grid, normal_fit, "r--", linewidth=2, label="normal, center k, spread √(2k)")
plt.title("χ² with many degrees of freedom approaches a normal")
plt.xlabel("total noise power")
plt.ylabel("Density")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()
print("5.  Nearly perfect fit — the central limit theorem: a sum of many "
      "independent variables (here Z²'s) becomes normal.")


## 8. Conclusion

- Noise power $W = Z^2$ of standard Gaussian noise follows the **chi-squared distribution with 1 degree of freedom**; its CDF is the Gaussian area between $\pm\sqrt{w}$.
- The PDF is the **slope of the CDF**: steep accumulation near 0 → density concentrated at small powers.
- Power over $k$ independent transmissions is $\chi^2_k = Z_1^2 + \cdots + Z_k^2$; larger $k$ spreads the distribution and makes it more symmetric.
- $\chi^2$ is a workhorse in statistics (variance estimation, goodness-of-fit tests) — here it emerged naturally from squaring the ubiquitous Gaussian.
